<a href="https://colab.research.google.com/github/801-Hillside-Terrace/SMART-2026/blob/main/week2/Week2_Ridge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1013]:
### v Don't modify any of this v ###

#imports
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import csv

# seed:
torch.manual_seed(801)

# import data:
url = 'https://raw.githubusercontent.com/801-Hillside-Terrace/SMART-2026/main/week2/mpg_train.csv'
mpg_data = pd.read_csv(url)


# split data:
y = mpg_data['mpg'].to_numpy()
X = mpg_data.drop(['mpg', 'origin'], axis=1).to_numpy()
X = torch.tensor(X, dtype=torch.float32)
X_np = X.clone()
noise = []
for i in range(5):
    base_col = X_np[:, i % X_np.shape[1]]
    new_col = base_col * (1 + 0.01 * torch.randn_like(base_col)) + 0.1 * torch.randn_like(base_col)
    noise.append(new_col.unsqueeze(1))

noise = torch.cat(noise, dim=1)

# concatenate
X = torch.cat([X, noise], dim=1)
y = torch.tensor(y, dtype=torch.float32).reshape(-1,1)
y = y + torch.randn_like(y) * 3.0



# some functions that may be helpful:

def train_val_split(X, y, val_pct=0.2, seed=801):
  # splits data into train and val based on val_pct
  torch.manual_seed(seed)
  n = X.shape[0]
  perm = torch.randperm(n)

  split = int(n * (1 - val_pct))
  train_idx = perm[:split]
  val_idx = perm[split:]

  return X[train_idx], X[val_idx], y[train_idx], y[val_idx]

def standardize_train_val(X_train, X_val):
  # standardizes features based on training means/stds
  mean = X_train.mean(dim=0, keepdim=True)
  std = X_train.std(dim=0, keepdim=True) + 1e-8

  X_train_scaled = (X_train - mean) / std
  X_val_scaled = (X_val - mean) / std

  return X_train_scaled, X_val_scaled, mean, std

def fit_ridge_model_GD(
    X_train, X_val, y_train, y_val,
    lambda_ = 0.0, lr = 0.01, epochs = 1000,
    seed = 801, verbose = False
):
# fits linear regression with Ridge loss using gradient descent
# no penalty on intercept
# lambda_ is lambda hyperparameter
  torch.manual_seed(seed)
  n = X_train.shape[0]
  criterion = nn.MSELoss()
  n_features = X_train.shape[1]
  model = nn.Linear(n_features, 1)
  optimizer = torch.optim.SGD(model.parameters(), lr=lr)
  train_losses = []
  val_losses = []

  for epoch in range(epochs):
    optimizer.zero_grad()
    model.train()

    y_hat = model(X_train)

    # multiplied by n to match if same lambda used for closed form
    loss = (n * criterion(y_hat, y_train)) + lambda_ * torch.sum(model.weight ** 2)

    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
      train_mse = criterion(model(X_train), y_train)
      val_mse = criterion(model(X_val), y_val)

    train_losses.append(train_mse.item())
    val_losses.append(val_mse.item())
  if verbose:
    return model, train_losses, val_losses
  else:
    return model, train_losses[-1], val_losses[-1]

def make_kfold_splits(X, y, k=5, seed=801):
  # makes k folds for cross validation
  torch.manual_seed(seed)
  n = X.shape[0]
  perm = torch.randperm(n)

  folds = torch.chunk(perm, k)

  splits = []

  for i in range(k):
    val_idx = folds[i]
    train_idx = torch.cat([folds[j] for j in range(k) if j != i])

    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]

    splits.append((X_train, X_val, y_train, y_val))

  return splits

def ols_with_intercept(X, y):
  # fits linear regression with MSE loss closed form (X doesnt need column of 1s)
  n = X.shape[0]

  ones = torch.ones((n, 1))
  X_aug = torch.cat([ones, X], dim=1)

  beta_full = torch.linalg.solve(X_aug.T @ X_aug, X_aug.T @ y).squeeze()

  beta0 = beta_full[0]
  beta = beta_full[1:]

  return beta0, beta

def ridge_with_intercept(X, y, lambda_):
  # fits linear regression with Ridge loss closed form (X doesnt need column of 1s)
  # no penalty on intercept
  # lambda_ is lambda hyperparameter
  n, p = X.shape

  ones = torch.ones((n, 1))
  X_aug = torch.cat([ones, X], dim=1)

  D = torch.eye(p + 1)
  D[0, 0] = 0

  beta_full = torch.linalg.solve(
        X_aug.T @ X_aug + lambda_ * D,
        X_aug.T @ y
    ).squeeze()

  beta0 = beta_full[0]
  beta = beta_full[1:]

  return beta0, beta

def predict(X, beta0, beta):
  # returns y_hat
  return (beta0 + X @ beta).unsqueeze(1)

def MSE(y, y_hat):
  # returns MSE
  return torch.mean((y - y_hat) ** 2)

def R2(y, y_hat):
  # returns R_Squared
  ss_res = torch.sum((y - y_hat) ** 2)
  ss_tot = torch.sum((y - y.mean()) ** 2)

  r2 = 1 - ss_res / ss_tot
  return r2

### ^ Don't modify any of this ^ ###

1. Using the provided $X$ and $y$ data, try to find an optimal $\lambda$ value by doing a train-validation split of 80-20 and trying different $\lambda$ values.

In [1014]:
# start with these and try to narrow it down from there:
lambdas = [0.0, 0.001, 0.01, 0.1, 1.0, 10.0, 15.0, 20.0, 25.0]

In [1015]:
#

Answer here:

2. Using the provided $X$ and $y$ data, try to find an optimal $\lambda$ value by doing 5-fold CV instead.

In [1016]:
#

Answer here:

3. Compare the beta values you got from (2) or (3) to those from using OLS/$\lambda=0$.

In [1017]:
#

Answer here:
